# Make a PDF from Python components

Edit an order, customize its cards and table, and export a PDF plus styled
and unstyled HTML. You write Python values and components; no HTML strings
are needed. All sample data is fictional.

**In Colab:** choose **Runtime → Run all**, sign in if prompted, and use a
standard CPU runtime. No GPU or paid plan is needed for this example;
[Colab availability and limits](https://research.google.com/colaboratory/faq.html)
still apply. Runtime files are temporary, so download your results before
leaving. Local Jupyter also works.

**Experimental:** this notebook pins Fullbleed **2.6.0a6**. The component API
may change and has documented CSS limits. Bootstrap components are static
in both PDF and HTML; no Bootstrap JavaScript is included.

## 1. Install the tested alpha

Run this before importing Fullbleed. If you already imported a different
version in this notebook session, restart its kernel/runtime first.
The wheel supplies the engine and font; Jupyter supplies display tools.

In [ ]:
%pip install --quiet --only-binary=:all: "fullbleed==2.6.0a6"

## 2. Make the order yours

Change a name, quantity or price, then rerun this cell and the cells below it.
Prices are decimal strings; this example omits tax, shipping and discounts.
Try `items=[]` to see the empty state, or add rows to explore pagination.

In [ ]:
order = {
    "number": "1048",
    "customer": "Moss & Fern Studio",
    "collection": "The autumn collection",
    "status": "Ready to ship",
    "items": [
        {"item": "Studio notebooks", "quantity": 24, "unit_price": "15.00"},
        {"item": "Cotton stationery sets", "quantity": 12, "unit_price": "18.00"},
        {"item": "Archival prints", "quantity": 8, "unit_price": "60.00"},
    ],
}

## 3. Choose the accent and type size

These values feed ordinary Python styles. Later, `.with_parts(...)` styles
a card's own title and body. The custom `dispatch-note` class gets its rule
from `Document(styles=...)`; a class name alone does not create a style.

In [ ]:
accent = "#205b47"
title_size = 30

## 4. Compose reusable components

`Panel` supplies reusable card defaults. `Metric` is an ordinary Python
function. `DataTable` builds its structure from rows and column functions.
Bootstrap classes such as `mb-4` come with the components; your custom
classes need their own rules. Text values are escaped automatically.

A column callback can return a component: `Strong(row["item"])` emphasizes
each item inside its table cell. Return text or compose other components
without constructing an HTML string.


In [ ]:
from decimal import Decimal
from fullbleed.ui.experimental import Document, Page, Strong, Theme, mm, pt, style, styled
from fullbleed.ui.experimental import bootstrap as bs

def money(value):
    return f"USD {value:,.2f}"

Panel = styled(bs.Card, border_color="#d7e1de", border_radius="8px")

def Metric(value, label):
    return Panel(
        bs.Heading(value, level=2, class_name="mb-1", style=style(color=accent)),
        bs.Text(label, class_name="text-secondary"),
    )

def OrderSummary(data):
    rows = [dict(row, amount=Decimal(row["unit_price"]) * row["quantity"])
            for row in data["items"]]
    total = sum((row["amount"] for row in rows), Decimal("0.00"))
    units = sum(row["quantity"] for row in rows)
    details = bs.DataTable(
        rows,
        columns=[
            bs.Column("Item", lambda row: Strong(row["item"])),
            bs.Column("Quantity", lambda row: row["quantity"], align="right"),
            bs.Column("Amount", lambda row: money(row["amount"]), align="right"),
        ],
        striped=True,
        caption="Order contents · USD",
    ) if rows else bs.Alert("No items in this order.", variant="light")
    return Document(
        bs.Box(bs.Text("NORTHSTAR / STUDIO", class_name="fw-bold"),
               bs.Badge(data["status"], variant="light", class_name="ms-3"),
               class_name="mb-4"),
        bs.Heading(data["collection"], level=1,
                   style=style(font_size=pt(title_size), color=accent, line_height=1.1)),
        bs.Paragraph(f'Prepared for {data["customer"]}', class_name="lead mb-4"),
        bs.Row(
            bs.Col(Metric(str(units), "Items in this order"), span=5),
            bs.Col(Metric(money(total), "Order total"), span=7),
            gutter=3, class_name="mb-4 mx-0",
        ),
        Panel(details, title=f'Order {data["number"]}',
              footer="Made with care. Packed with purpose.").with_parts(
            title=style(color=accent, font_size=pt(16)),
            body=style(padding="1.25rem"),
        ),
        bs.Alert("Please retain this copy with your shipment.",
                 class_name="dispatch-note mt-4"),
        title=f'Order {data["number"]}',
        page=Page(size="A4", margin=mm(16)),
        theme=Theme(text=style(font_family="Inter, Arial, sans-serif", font_size=pt(10),
                               color="#18382f", line_height=1.4)),
        styles={".dispatch-note": style(color=accent, border_color=accent,
                                        background_color="#f0f5f2")},
    )

document = OrderSummary(order)

## 5. Render and export

The PDF renderer receives the bundled Inter font file explicitly. A CSS
family name does not load a font. Styled HTML keeps the family declaration;
its browser host supplies Inter separately, or uses the Arial/sans-serif fallback.

`include_styles=False` removes generated and inline styling but keeps classes
for a host application's CSS. Both export modes also work on an individual
component, such as `Metric("44", "Items").to_html(include_styles=False)`.
Styled Bootstrap fragments include global Bootstrap rules; use unstyled
fragments when the host page already provides its stylesheet.

In [ ]:
from importlib.resources import files
from pathlib import Path
from fullbleed.ui.experimental import Renderer

output = Path("output")
output.mkdir(exist_ok=True)
font = files("fullbleed_assets").joinpath("fonts/Inter-Variable.ttf")
renderer = Renderer(font_files=[str(font)],
                    document_timestamp="2026-10-01T00:00:00Z")
prepared = document.prepare(renderer)
pdf_path = output / "order.pdf"
pdf_path.write_bytes(prepared.render_pdf())
(output / "order.styled.html").write_text(
    document.to_html(include_styles=True), encoding="utf-8", newline="\n")
(output / "order.unstyled.html").write_text(
    document.to_html(include_styles=False), encoding="utf-8", newline="\n")
print(f"Saved {pdf_path} and both HTML exports.")

In [ ]:
from IPython.display import Image, display

preview_paths = []
for number, png in enumerate(prepared.render_image_pages(dpi=110), start=1):
    path = output / f"order-page-{number}.png"
    path.write_bytes(png)
    preview_paths.append(path)
    display(Image(data=png, width=650))
print(f"{len(preview_paths)} page(s). Review every page before using the document.")

## 6. Download your files

This ZIP contains the current PDF, both HTML exports, and every current page
preview. In Colab the next cell starts the download. In local Jupyter it
displays a link. The runtime's `output/` folder also contains the individual
files. If you edit the order or styles, rerun from that edit through this cell.

In [ ]:
import sys
from zipfile import ZipFile, ZIP_DEFLATED

bundle_path = output / "component-order.zip"
current_files = [pdf_path, output / "order.styled.html",
                 output / "order.unstyled.html", *preview_paths]
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
    for path in current_files:
        bundle.write(path, arcname=path.name)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(bundle_path))
else:
    from IPython.display import FileLink
    display(FileLink(str(bundle_path)))

## Keep composing

- [Component guide and catalog](https://docs.fullbleed.dev/guides/bootstrap-components-preview/)
  explain class names, named parts, fonts, static states and current limits.
- [Northstar and Common Ground source kits](https://docs.fullbleed.dev/guides/showcase-authoring/)
  show a complete invoice and illustrated report in components and HTML/CSS.
- [The authored accessible guide](https://docs.fullbleed.dev/guides/accessible-showcase/)
  includes the exact PDF, semantic source and scoped validation receipts.
- [Share an authoring example](https://github.com/fullbleed-engine/fullbleed-official/discussions/86)
  if a composition or styling task feels awkward. Include the small component,
  expected result and Fullbleed version; use fictional data in public posts.
- [Stable Python guide](https://docs.fullbleed.dev/guides/python-components/)
  documents the established API separately from this alpha.

The notebook and Fullbleed are MIT licensed. This example does not establish
complete Bootstrap CSS, browser, accessibility or PDF conformance.